# T4D Response V2 - Phase 4 Chemistry Response And Scenarios

Goal: fit simple station-year chemistry response models. Keep salinity and pH annual. Let oxygen use warm-season summaries. Then run a small benchmark scenario set.

Outputs:
- `t4d.v2.phase4.station_year_model_table.csv`
- `t4d.v2.phase4.response_scores.csv`
- `t4d.v2.phase4.holdout_predictions.csv`
- `t4d.v2.phase4.scenario_summary.csv`
- `t4d.v2.phase4.scenario_by_region.csv`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme( style = 'whitegrid' )
pd.set_option( 'display.max_columns', 160 )

phase_dir = Path( '../data/reference/response_v2' )
daily_path = phase_dir / 't4d.v2.phase3.daily_with_temp_pred.csv'
cohort_path = phase_dir / 't4d.v2.phase2.cohort_keys.csv'

station_year_out_path = phase_dir / 't4d.v2.phase4.station_year_model_table.csv'
scores_out_path = phase_dir / 't4d.v2.phase4.response_scores.csv'
holdout_pred_out_path = phase_dir / 't4d.v2.phase4.holdout_predictions.csv'
scenario_out_path = phase_dir / 't4d.v2.phase4.scenario_summary.csv'
scenario_region_out_path = phase_dir / 't4d.v2.phase4.scenario_by_region.csv'

daily = pd.read_csv( daily_path, parse_dates = [ 'date' ] )
cohort_keys = pd.read_csv( cohort_path )
daily = daily.merge( cohort_keys[ [ 'station_key', 'cohort' ] ], on = [ 'station_key', 'cohort' ], how = 'left' )

In [ ]:
# Build one station-year table.
# Keep annual summaries for salinity and pH.
# Keep warm-season summaries for oxygen.
daily[ 'year' ] = daily[ 'date' ].dt.year
daily[ 'month_num' ] = daily[ 'date' ].dt.month
daily[ 'is_warm_season' ] = daily[ 'month_num' ].between( 6, 9 )

annual_feature_cols = [ 'delta_water_temp_pred', 'air_temp', 'precipitation', 'wind_speed', 'solar_radiation' ]
annual_feature_cols = [ col for col in annual_feature_cols if col in daily.columns ]

annual_agg = { 'n_days_annual': ( 'date', 'size' ) }
for col in annual_feature_cols:
    annual_agg[ f'{ col }_mean' ] = ( col, 'mean' )
    annual_agg[ f'{ col }_std' ] = ( col, 'std' )
for target in [ 'delta_salinity', 'delta_oxygen', 'delta_ph' ]:
    if target in daily.columns:
        annual_agg[ f'{ target }_annual_mean' ] = ( target, 'mean' )

station_year = ( 
    daily
    .groupby( [ 'station_key', 'station_code', 'region', 'region_name', 'station_name', 'year', 'cohort' ], as_index = False )
    .agg( **annual_agg )
)

warm = daily.loc[ daily[ 'is_warm_season' ] ].copy( )
warm_agg = { 'n_days_warm': ( 'date', 'size' ) }
for col in annual_feature_cols:
    warm_agg[ f'{ col }_warm_mean' ] = ( col, 'mean' )
    warm_agg[ f'{ col }_warm_std' ] = ( col, 'std' )
for target in [ 'delta_salinity', 'delta_oxygen', 'delta_ph' ]:
    if target in warm.columns:
        warm_agg[ f'{ target }_warm_mean' ] = ( target, 'mean' )

warm_station_year = ( 
    warm
    .groupby( [ 'station_key', 'year' ], as_index = False )
    .agg( **warm_agg )
)
station_year = station_year.merge( warm_station_year, on = [ 'station_key', 'year' ], how = 'left' )

station_year[ 'keep_oxygen_row' ] = ( 
    station_year[ 'n_days_warm' ].fillna( 0 ) >= 45
)

print( 'station-year model rows:', len( station_year ) )
display( station_year.head( 8 ) )

In [ ]:
# Fit one simple model stack per target.
# Oxygen uses the warm-season target and warm-season features.
# Salinity and pH keep the annual target and annual features.
target_setup = { 
    'delta_salinity': { 
        'target_col': 'delta_salinity_annual_mean',
        'feature_cols': [ 
            'delta_water_temp_pred_mean',
            'air_temp_mean',
            'precipitation_mean',
            'wind_speed_mean',
            'solar_radiation_mean',
            'delta_water_temp_pred_std',
            'air_temp_std',
            'precipitation_std',
            'wind_speed_std',
            'solar_radiation_std',
        ],
        'row_mask_col': None,
        'window_label': 'annual mean',
    },
    'delta_ph': { 
        'target_col': 'delta_ph_annual_mean',
        'feature_cols': [ 
            'delta_water_temp_pred_mean',
            'air_temp_mean',
            'precipitation_mean',
            'wind_speed_mean',
            'solar_radiation_mean',
            'delta_water_temp_pred_std',
            'air_temp_std',
            'precipitation_std',
            'wind_speed_std',
            'solar_radiation_std',
        ],
        'row_mask_col': None,
        'window_label': 'annual mean',
    },
    'delta_oxygen': { 
        'target_col': 'delta_oxygen_warm_mean',
        'feature_cols': [ 
            'delta_water_temp_pred_warm_mean',
            'air_temp_warm_mean',
            'precipitation_warm_mean',
            'wind_speed_warm_mean',
            'solar_radiation_warm_mean',
            'delta_water_temp_pred_warm_std',
            'air_temp_warm_std',
            'precipitation_warm_std',
            'wind_speed_warm_std',
            'solar_radiation_warm_std',
        ],
        'row_mask_col': 'keep_oxygen_row',
        'window_label': 'warm-season mean',
    },
}

score_rows = [ ]
holdout_rows = [ ]
model_store = { }
feature_store = { }
fill_store = { }

for target, setup in target_setup.items( ):
    target_col = setup[ 'target_col' ]
    feature_cols = [ col for col in setup[ 'feature_cols' ] if col in station_year.columns ]
    row_mask_col = setup[ 'row_mask_col' ]

    model_df = station_year.copy( )
    if row_mask_col is not None:
        model_df = model_df.loc[ model_df[ row_mask_col ].fillna( False ) ].copy( )

    train_df = model_df.loc[ model_df[ 'cohort' ] == 'train' ].dropna( subset = [ target_col ] ).copy( )
    holdout_df = model_df.loc[ model_df[ 'cohort' ] == 'holdout' ].dropna( subset = [ target_col ] ).copy( )

    if len( train_df ) == 0 or len( holdout_df ) == 0 or len( feature_cols ) == 0:
        continue

    X_train = train_df[ feature_cols ].copy( )
    X_holdout = holdout_df[ feature_cols ].copy( )
    fill_values = X_train.median( numeric_only = True )
    X_train = X_train.fillna( fill_values )
    X_holdout = X_holdout.fillna( fill_values )

    y_train = train_df[ target_col ]
    y_holdout = holdout_df[ target_col ]

    pred_naive = np.full( len( holdout_df ), float( y_train.median( ) ) )

    ridge_model = Ridge( alpha = 1.0 )
    ridge_model.fit( X_train, y_train )
    pred_ridge = ridge_model.predict( X_holdout )

    hgb_model = HistGradientBoostingRegressor( learning_rate = 0.05, max_depth = 4, max_iter = 300, min_samples_leaf = 20, random_state = 42 )
    hgb_model.fit( X_train, y_train )
    pred_hgb = hgb_model.predict( X_holdout )

    model_scores = [ 
        ( 'naive_global_median', pred_naive ),
        ( 'ridge_phase7_climate', pred_ridge ),
        ( 'hgb_phase7_climate', pred_hgb ),
    ]
    for model_name, pred in model_scores:
        score_rows.append( { 
            'target': target,
            'target_window': setup[ 'window_label' ],
            'model': model_name,
            'mae': float( mean_absolute_error( y_holdout, pred ) ),
            'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
            'r2': float( r2_score( y_holdout, pred ) ),
            'n_train': int( len( train_df ) ),
            'n_holdout': int( len( holdout_df ) ),
            'n_features': int( len( feature_cols ) ),
        } )

    selected_name = min( 
        [ ( 'ridge_phase7_climate', pred_ridge, ridge_model ), ( 'hgb_phase7_climate', pred_hgb, hgb_model ) ],
        key = lambda item: mean_squared_error( y_holdout, item[ 1 ] ) ** 0.5,
    )[ 0 ]
    selected_model = ridge_model if selected_name == 'ridge_phase7_climate' else hgb_model
    selected_pred = pred_ridge if selected_name == 'ridge_phase7_climate' else pred_hgb

    model_store[ target ] = selected_model
    feature_store[ target ] = feature_cols
    fill_store[ target ] = fill_values

    tmp = holdout_df[ [ 'station_key', 'station_code', 'region', 'region_name', 'station_name', 'year', 'cohort' ] ].copy( )
    tmp[ 'target' ] = target
    tmp[ 'target_window' ] = setup[ 'window_label' ]
    tmp[ 'y_true' ] = y_holdout.values
    tmp[ 'y_pred' ] = selected_pred
    tmp[ 'residual' ] = tmp[ 'y_pred' ] - tmp[ 'y_true' ]
    holdout_rows.append( tmp )

response_scores = pd.DataFrame( score_rows ).sort_values( [ 'target', 'rmse' ] ).reset_index( drop = True )
holdout_predictions = pd.concat( holdout_rows, ignore_index = True )

print( 'phase 4 response scores:' )
display( response_scores.round( 4 ) )

In [ ]:
# Keep the diagnostics simple and visible.
fig, axes = plt.subplots( 1, 2, figsize = ( 16, 5 ) )

plot_scores = response_scores.loc[ response_scores[ 'model' ] != 'naive_global_median' ].copy( )
plot_scores[ 'target_label' ] = plot_scores[ 'target' ] + ' (' + plot_scores[ 'target_window' ] + ')'
sns.barplot( data = plot_scores, x = 'target_label', y = 'rmse', hue = 'model', ax = axes[ 0 ] )
axes[ 0 ].set_title( 'Phase 4 Holdout RMSE By Target' )
axes[ 0 ].set_xlabel( 'Target' )
axes[ 0 ].set_ylabel( 'RMSE' )
axes[ 0 ].tick_params( axis = 'x', rotation = 20 )

holdout_predictions[ 'target_label' ] = holdout_predictions[ 'target' ] + ' (' + holdout_predictions[ 'target_window' ] + ')'
sns.boxplot( data = holdout_predictions, x = 'target_label', y = 'residual', ax = axes[ 1 ] )
axes[ 1 ].axhline( 0, color = 'black', linestyle = '--', linewidth = 1.0 )
axes[ 1 ].set_title( 'Phase 4 Holdout Residual Spread' )
axes[ 1 ].set_xlabel( 'Target' )
axes[ 1 ].set_ylabel( 'Predicted - observed' )
axes[ 1 ].tick_params( axis = 'x', rotation = 20 )

plt.tight_layout( )
plt.show( )

In [ ]:
# Run a small benchmark scenario set.
# Re-use the fitted response models.
# Keep the scenario edits blunt and easy to explain.
scenarios = pd.DataFrame( [ 
    { 'scenario': 'baseline_ref', 'air_temp_add_c': 0.0, 'precip_mult': 1.00 },
    { 'scenario': 'plus1p0C_precip_same', 'air_temp_add_c': 1.0, 'precip_mult': 1.00 },
    { 'scenario': 'plus1p5C_precip_same', 'air_temp_add_c': 1.5, 'precip_mult': 1.00 },
    { 'scenario': 'plus2p0C_precip_same', 'air_temp_add_c': 2.0, 'precip_mult': 1.00 },
] )

scenario_rows = [ ]
scenario_region_rows = [ ]

for target, model in model_store.items( ):
    feature_cols = feature_store[ target ]
    fill_values = fill_store[ target ]
    setup = target_setup[ target ]

    model_df = station_year.copy( )
    if setup[ 'row_mask_col' ] is not None:
        model_df = model_df.loc[ model_df[ setup[ 'row_mask_col' ] ].fillna( False ) ].copy( )

    X_base = model_df[ feature_cols ].copy( ).fillna( fill_values )

    for _, scen in scenarios.iterrows( ):
        X_scen = X_base.copy( )

        for col in [ 'air_temp_mean', 'air_temp_warm_mean' ]:
            if col in X_scen.columns:
                X_scen[ col ] = X_scen[ col ] + float( scen[ 'air_temp_add_c' ] )

        for col in [ 'precipitation_mean', 'precipitation_warm_mean' ]:
            if col in X_scen.columns:
                X_scen[ col ] = X_scen[ col ] * float( scen[ 'precip_mult' ] )

        for col in [ 'delta_water_temp_pred_mean', 'delta_water_temp_pred_warm_mean' ]:
            if col in X_scen.columns:
                X_scen[ col ] = X_scen[ col ] + float( scen[ 'air_temp_add_c' ] )

        pred = model.predict( X_scen )
        scenario_name = str( scen[ 'scenario' ] )

        scenario_rows.append( { 
            'target': target,
            'target_window': setup[ 'window_label' ],
            'scenario': scenario_name,
            'mean_pred': float( np.mean( pred ) ),
            'median_pred': float( np.median( pred ) ),
            'n_station_years': int( len( pred ) ),
        } )

        region_tmp = model_df.assign( pred = pred ).groupby( 'region', as_index = False )[ 'pred' ].mean( )
        for _, row in region_tmp.iterrows( ):
            scenario_region_rows.append( { 
                'target': target,
                'target_window': setup[ 'window_label' ],
                'scenario': scenario_name,
                'region': row[ 'region' ],
                'mean_pred': float( row[ 'pred' ] ),
            } )

scenario_summary = pd.DataFrame( scenario_rows )
scenario_baseline = ( 
    scenario_summary
    .loc[ scenario_summary[ 'scenario' ] == 'baseline_ref', [ 'target', 'mean_pred' ] ]
    .rename( columns = { 'mean_pred': 'baseline_mean_pred' } )
)
scenario_summary = scenario_summary.merge( scenario_baseline, on = 'target', how = 'left' )
scenario_summary[ 'delta_vs_baseline' ] = scenario_summary[ 'mean_pred' ] - scenario_summary[ 'baseline_mean_pred' ]

scenario_by_region = pd.DataFrame( scenario_region_rows )
scenario_region_baseline = ( 
    scenario_by_region
    .loc[ scenario_by_region[ 'scenario' ] == 'baseline_ref', [ 'target', 'region', 'mean_pred' ] ]
    .rename( columns = { 'mean_pred': 'baseline_mean_pred' } )
)
scenario_by_region = scenario_by_region.merge( scenario_region_baseline, on = [ 'target', 'region' ], how = 'left' )
scenario_by_region[ 'delta_vs_baseline' ] = scenario_by_region[ 'mean_pred' ] - scenario_by_region[ 'baseline_mean_pred' ]

plot_df = scenario_summary.loc[ scenario_summary[ 'scenario' ] != 'baseline_ref' ].copy( )
plot_df[ 'target_label' ] = plot_df[ 'target' ] + ' (' + plot_df[ 'target_window' ] + ')'

plt.figure( figsize = ( 12, 5 ) )
sns.barplot( data = plot_df, x = 'target_label', y = 'delta_vs_baseline', hue = 'scenario' )
plt.axhline( 0, color = 'black', linestyle = '--', linewidth = 1.0 )
plt.title( 'Phase 4 Scenario Shift By Target' )
plt.xlabel( 'Target' )
plt.ylabel( 'Delta vs baseline' )
plt.xticks( rotation = 20 )
plt.tight_layout( )
plt.show( )

In [ ]:
station_year.to_csv( station_year_out_path, index = False )
response_scores.to_csv( scores_out_path, index = False )
holdout_predictions.to_csv( holdout_pred_out_path, index = False )
scenario_summary.to_csv( scenario_out_path, index = False )
scenario_by_region.to_csv( scenario_region_out_path, index = False )

print( f'saved: {station_year_out_path}' )
print( f'saved: {scores_out_path}' )
print( f'saved: {holdout_pred_out_path}' )
print( f'saved: {scenario_out_path}' )
print( f'saved: {scenario_region_out_path}' )